# 08 - Cross-dataset evaluation: Kaggle test vs NIH ChestX-ray14

**Group 5, ICSI 535.** Loads the three saved models (Dense, CNN, ResNet50) and the **decision thresholds already chosen on Kaggle validation** (`results/selected_thresholds_validation.csv`, written by notebook 07). It then measures the same models, with the same thresholds, on:

1. the **Kaggle test set** (same hospital/population as training), and
2. the **NIH ChestX-ray14 external set** (adult patients, different labels), built by `scripts/make_nih_dataset.py`.

**Rules:** no model is retrained and no threshold is re-tuned on test or NIH data. Any drop from Kaggle to NIH is the generalization result.

**Inputs:** `data/processed_224_pad_gray/test.npz`, `data/processed_224_pad_gray_nih/test.npz`, `models/*.keras`, `results/selected_thresholds_validation.csv`.
**Outputs:** `results/cross_dataset_metrics.csv`, `results/cross_dataset_roc.png`, `results/cross_dataset_calibration.png`, `results/nih_predictions.csv`.

> **Limits to state:** (a) 170 Kaggle test patient IDs also appear in train/validation, and we cannot confirm whether they are the same people; (b) NIH labels are extracted automatically from radiology reports (noisier); (c) Kaggle is pediatric, NIH is adult; (d) "Pneumonia" images in NIH may also carry other findings. This is a course experiment, **not clinical evidence**.

## 1. Settings and paths

In [ ]:
from pathlib import Path
import gc, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, brier_score_loss)
from sklearn.calibration import calibration_curve

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
KAGGLE_DIR = PROJECT_DIR / "data" / "processed_224_pad_gray"
NIH_DIR = PROJECT_DIR / "data" / "processed_224_pad_gray_nih"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8
SEED = 42
N_BOOT = 1000                    # bootstrap resamples for the AUC confidence interval
EVALUATE_KAGGLE_TEST = True      # set False to run NIH only
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__, "| Project:", PROJECT_DIR)
need = [NIH_DIR / "test.npz", RESULTS_DIR / "selected_thresholds_validation.csv",
        MODEL_DIR / "dense_baseline.keras", MODEL_DIR / "cnn_from_scratch.keras", MODEL_DIR / "resnet50_transfer.keras"]
if EVALUATE_KAGGLE_TEST:
    need.append(KAGGLE_DIR / "test.npz")
for p in need:
    assert p.exists(), f"Missing: {p}"
print("All inputs found.")

## 2. Load the evaluation sets (uint8 images, labels 0 = normal, 1 = pneumonia)

In [ ]:
def load_npz(path):
    with np.load(path) as a:
        X = a["images"]; y = a["labels"].astype(np.int32)
    assert X.ndim == 3 and X.shape[1:] == (224, 224) and X.dtype == np.uint8
    assert len(X) == len(y) and set(np.unique(y)) <= {0, 1}
    return X, y

datasets = {}
if EVALUATE_KAGGLE_TEST:
    datasets["Kaggle test"] = load_npz(KAGGLE_DIR / "test.npz")
datasets["NIH external"] = load_npz(NIH_DIR / "test.npz")
for name, (X, y) in datasets.items():
    print(f"{name}: {X.shape}, normal={int((y==0).sum())}, pneumonia={int((y==1).sum())}")

## 3. Prepare inputs exactly as in notebook 07 and load the frozen thresholds

In [ ]:
def make_ds(X, kind):
    ds = tf.data.Dataset.from_tensor_slices(X).batch(BATCH_SIZE)
    def process(batch):
        batch = tf.cast(batch, tf.float32)
        if kind == "dense":
            return batch / 255.0
        if kind == "cnn":
            return tf.expand_dims(batch / 255.0, -1)
        if kind == "resnet":
            batch = tf.repeat(tf.expand_dims(batch, -1), repeats=3, axis=-1)
            return keras.applications.resnet50.preprocess_input(batch)
        raise ValueError(kind)
    return ds.map(process, num_parallel_calls=1).prefetch(1)

models_to_run = {
    "Dense baseline":    (MODEL_DIR / "dense_baseline.keras", "dense"),
    "CNN from scratch":  (MODEL_DIR / "cnn_from_scratch.keras", "cnn"),
    "ResNet50 transfer": (MODEL_DIR / "resnet50_transfer.keras", "resnet"),
}
th = pd.read_csv(RESULTS_DIR / "selected_thresholds_validation.csv").set_index("model")["threshold"].to_dict()
assert set(th) == set(models_to_run), f"Threshold names {set(th)} do not match models"
print("Frozen thresholds (chosen on Kaggle validation):")
for k, v in th.items():
    print(f"  {k}: {v:.4f}")

## 4. Predict (one model at a time, to limit memory)

In [ ]:
probs = {ds_name: {} for ds_name in datasets}
for model_name, (path, kind) in models_to_run.items():
    print(f"\nLoading {model_name}")
    model = keras.models.load_model(path, compile=False)
    for ds_name, (X, y) in datasets.items():
        p = model.predict(make_ds(X, kind), verbose=0).reshape(-1).astype(np.float64)
        assert len(p) == len(y) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
        probs[ds_name][model_name] = p
        print(f"  {ds_name}: done (mean probability {p.mean():.3f})")
    del model
    keras.backend.clear_session(); gc.collect()

## 5. Metrics at the frozen thresholds

Sensitivity = pneumonia found; specificity = normal correctly cleared. The AUC gets a 95% bootstrap interval so small differences are not over-read.

In [ ]:
rng = np.random.default_rng(SEED)

def auc_ci(y, p, n=N_BOOT):
    vals = []
    idx = np.arange(len(y))
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[s])) < 2:
            continue
        vals.append(roc_auc_score(y[s], p[s]))
    return np.percentile(vals, [2.5, 97.5])

rows = []
for ds_name, (X, y) in datasets.items():
    for model_name in models_to_run:
        p = probs[ds_name][model_name]
        t = th[model_name]
        pred = (p >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        lo, hi = auc_ci(y, p)
        rows.append({"dataset": ds_name, "model": model_name, "threshold": t,
                     "roc_auc": roc_auc_score(y, p), "auc_ci_low": lo, "auc_ci_high": hi,
                     "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                     "brier_score": brier_score_loss(y, p),
                     "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
metrics = pd.DataFrame(rows)
metrics.to_csv(RESULTS_DIR / "cross_dataset_metrics.csv", index=False)
display(metrics.round(4))

## 6. Compare with Kaggle validation (the numbers on the midterm slide)

In [ ]:
val_path = RESULTS_DIR / "model_comparison_validation.csv"
if val_path.exists():
    v = pd.read_csv(val_path)
    v = v[v["threshold_policy"].str.contains(">=95")][["model", "roc_auc", "sensitivity", "specificity", "brier_score"]].copy()
    v.insert(0, "dataset", "Kaggle validation")
    table = pd.concat([v, metrics[["dataset", "model", "roc_auc", "sensitivity", "specificity", "brier_score"]]], ignore_index=True)
else:
    table = metrics[["dataset", "model", "roc_auc", "sensitivity", "specificity", "brier_score"]]
order = {"Kaggle validation": 0, "Kaggle test": 1, "NIH external": 2}
table = table.sort_values(by=["model", "dataset"], key=lambda c: c.map(order) if c.name == "dataset" else c)
display(table.round(4).reset_index(drop=True))

## 7. ROC curves: same models, different datasets

In [ ]:
n = len(datasets)
fig, axes = plt.subplots(1, n, figsize=(6.2 * n, 5.4), squeeze=False)
for ax, (ds_name, (X, y)) in zip(axes[0], datasets.items()):
    for model_name in models_to_run:
        p = probs[ds_name][model_name]
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={roc_auc_score(y, p):.3f})")
        t = th[model_name]
        pred = (p >= t)
        ax.scatter([((pred == 1) & (y == 0)).sum() / (y == 0).sum()], [((pred == 1) & (y == 1)).sum() / (y == 1).sum()], s=40, zorder=5)
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set_title(f"{ds_name} (dots = frozen thresholds)")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.legend(loc="lower right"); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cross_dataset_roc.png", dpi=160)
plt.show()

## 8. Calibration: do predicted probabilities still match reality?

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(6.2 * n, 5.4), squeeze=False)
for ax, (ds_name, (X, y)) in zip(axes[0], datasets.items()):
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")
    for model_name in models_to_run:
        frac, mean_pred = calibration_curve(y, probs[ds_name][model_name], n_bins=10, strategy="uniform")
        ax.plot(mean_pred, frac, marker="o", label=model_name)
    ax.set_title(ds_name + " calibration")
    ax.set_xlabel("Mean predicted pneumonia probability"); ax.set_ylabel("Observed fraction pneumonia")
    ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cross_dataset_calibration.png", dpi=160)
plt.show()

## 9. Save predictions and print the slide summary

In [ ]:
nih_X, nih_y = datasets["NIH external"]
pred_df = pd.DataFrame({"nih_row_index": np.arange(len(nih_y)), "true_label": nih_y})
for model_name in models_to_run:
    key = {"Dense baseline": "dense", "CNN from scratch": "cnn", "ResNet50 transfer": "resnet50"}[model_name]
    pred_df[f"{key}_prob_pneumonia"] = probs["NIH external"][model_name]
pred_df.to_csv(RESULTS_DIR / "nih_predictions.csv", index=False)

print("SLIDE: cross-dataset result (frozen Kaggle-validation thresholds, no re-tuning)")
for model_name in models_to_run:
    line = f"- {model_name}: "
    parts = []
    for ds_name in datasets:
        r = metrics[(metrics.dataset == ds_name) & (metrics.model == model_name)].iloc[0]
        parts.append(f"{ds_name} AUC {r.roc_auc:.3f}, sens {r.sensitivity:.1%}, spec {r.specificity:.1%}")
    print(line + " | ".join(parts))
print("\nFiles saved in results/: cross_dataset_metrics.csv, cross_dataset_roc.png, cross_dataset_calibration.png, nih_predictions.csv")

## Interpretation caution

- A drop from Kaggle to NIH is expected: different population (adult vs pediatric), different labelling, different scanners. It shows how much the models rely on dataset-specific features; it is not a verdict on clinical usefulness.
- Both datasets use "Pneumonia vs normal / No Finding" as a proxy task. Label meanings are not identical.
- The Kaggle test set has possible patient-ID overlap with train/validation (170 IDs); treat Kaggle test numbers with that caveat.
- Compare AUC with its interval, then sensitivity and specificity at the frozen thresholds. Do not choose a new threshold from NIH results.